In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import mannwhitneyu, shapiro, kstest,f_oneway,kruskal, norm, sem
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, confusion_matrix,recall_score,roc_curve, auc
from itertools import cycle
import seaborn as sns
import random
from itertools import chain 
from sklearn.linear_model import Lasso


# functions

In [ ]:
def scaling_robust (x_train, x_test):
    # fit scaler on training data
    scaler = RobustScaler().fit(x_train)
    # apply scaler on training and test data
    x_train_scaled = pd.DataFrame(scaler.transform(x_train))
    x_test_scaled = pd.DataFrame(scaler.transform(x_test))
    x_train_scaled.columns = x_train.columns
    x_test_scaled.columns = x_train.columns
    
    return (x_train_scaled,x_test_scaled)

In [ ]:
def resampling_smote (x_train,y_train):
    x_train_resampled, y_train_resampled = SMOTE(random_state=3,sampling_strategy='not majority' ).fit_resample(x_train, y_train) 
    return (x_train_resampled,y_train_resampled)

In [ ]:
# mann whitney U-test - binary task
def feat_sel_mw (x_train,y_train,x_test):
    # training set per class
    y_train.reset_index(drop=True, inplace = True)
    class0 = x_train.loc[y_train == 0]
    class1 = x_train.loc[y_train == 1]
    # perform test
    U1, pv = mannwhitneyu(class0, class1)
    # select feature with pvalue<0.01
    train_selected_features_mw = x_train.iloc[:,pv<0.01]
    test_selected_features_mw = x_test.iloc[:,pv<0.01]
    # if no feature with pvalue<0.01 -> take the first 30 with smallest pvalue
    if train_selected_features_mw.shape[1] == 0:
        #pvalue descending order
        index_min_pv = np.argsort(-pv)
        # create final df
        train_selected_features_mw = pd.DataFrame()
        test_selected_features_mw = pd.DataFrame()
        for i in range(31):
            col_i = x_train.columns[index_min_pv[i]]
            train_selected_features_mw[col_i] = x_train[[col_i]]
            test_selected_features_mw[col_i] = x_test[[col_i]]

    return (train_selected_features_mw,test_selected_features_mw)

In [ ]:
# Kruskal-Wallis test or one-way ANOVA test - multiclass task
def feat_sel_stat (x_train, y_train, x_test):
    y_train.reset_index(drop=True, inplace = True)
    class0 = x_train.loc[y_train == 0]
    class1 = x_train.loc[y_train == 1]
    class2 = x_train.loc[y_train == 2]
    pvf = []
#small sample -> shapiro normality test
    if len(y_train) < 50: 
        S0, pv0 = shapiro(class0)
        S1, pv1 = shapiro(class1)
        S2, pv2 = shapiro(class2)
#normality -> one way anova test
        if min(pv0,pv1,pv2)>0.01: 
            for st in range(np.shape(class0)[1]):
                sa,pva = f_oneway(class0.iloc[:,st],class1.iloc[:,st],class2.iloc[:,st])
                pvf.append(pva)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#non-normality -> kruscal wallis test
        else: 
            sk,pvf = kruskal(class0,class1,class2)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#bigger sample -> kolmogorov smirnov normality test
    else:
        pv = []
        for r in range(np.shape(class0)[1]):
            res0 = kstest(class0.iloc[:,r], 'norm')
            pv.append(res0[1])
            res1 = kstest(class1.iloc[:,r],'norm')
            pv.append(res1[1])
            res2 = kstest(class2.iloc[:,r],'norm')
            pv.append(res2[1])
#normality -> one way anova test
        if min(pv)>0.01: 
            for st in range(np.shape(class0)[1]):
                sa,pva = f_oneway(class0.iloc[:,st],class1.iloc[:,st],class2.iloc[:,st])
                pvf.append(pva)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#non-normality -> kruscal wallis test
        else:
            sk,pvf = kruskal(class0,class1,class2)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]

    return (x_train_selected_stat,x_test_selected_stat)

In [ ]:
def feat_sel_lasso (x_train,y_train,x_test, alpha): 
    # Fit Lasso model
    lasso = Lasso(alpha=alpha)
    lasso.fit(x_train, y_train)
    # Select features with non-zero coefficients
    selected_features = x_train.columns[(lasso.coef_ != 0)]
    # Create new DataFrames with selected features
    x_train_selected = x_train[selected_features]
    x_test_selected = x_test[selected_features]

    return (x_train_selected,x_test_selected)

In [ ]:
# feature selection binary task
def feat_sel(x_train, x_test, y_train, alpha):
    x_train_mw, x_test_mw = feat_sel_mw (x_train,y_train,x_test)
    n_feat_stat = np.shape(x_train_mw)[1]

    x_train_mw_lasso,x_test_mw_lasso = feat_sel_lasso(x_train_mw,y_train,x_test_mw,alpha)
    n_feat = np.shape(x_train_mw_lasso)[1]

    print(f'binary n feat stat: {n_feat_stat} - n feat sel lasso: {n_feat} - alpha:{alpha}')    
    return (x_train_mw_lasso,x_test_mw_lasso)


In [ ]:
# feature selection multiclass task
def feat_sel_multi(x_train, x_test, y_train, alpha):
    x_train_stat, x_test_stat = feat_sel_stat (x_train, y_train, x_test)
    n_feat_stat = np.shape(x_train_stat)[1]

    x_train_stat_lasso, x_test_stat_lasso = feat_sel_lasso(x_train_stat, y_train, x_test_stat, alpha)
    n_feat = np.shape(x_train_stat_lasso)[1]

    print(f'0-1234-5 n feat stat: {n_feat_stat} - n feat sel lasso: {n_feat} - alpha:{alpha}') 
    return (x_train_stat_lasso, x_test_stat_lasso)

In [ ]:
def training_set_creation(train_pat,training_idx,patients,X,y):
    #train_pat is a df with 2 columns: pat_lis (id), imgXcor (number of images per coronary (LAD,LCX,RCA))
    i = 0
    tr = train_pat.iloc[training_idx].pat_list # id patients training crossvalidation
    tr.reset_index(drop=True, inplace=True)
    for pt in range(len(tr)):
        pat_tr = tr.iloc[pt]#id patient
        indices_tr = [i for i, s in enumerate(patients) if pat_tr in s]# trainig indices in X and y corresponding to selected patient 
        x_rows_cor_tr = X.iloc[indices_tr]# X rows in training
        y_rows_cor_tr = y.iloc[indices_tr]# y rows in training
        if i == 0:#first patient -> initialize
            x_train_fold = x_rows_cor_tr
            y_train_fold = y_rows_cor_tr
            i = 1
        else:    # from the second patient on -> concatente
            x_train_fold = pd.concat([x_train_fold,x_rows_cor_tr]) 
            y_train_fold = pd.concat([y_train_fold,y_rows_cor_tr])
    return (x_train_fold,y_train_fold)



In [ ]:
def validation_set_creation (train_pat, y_train_pat, validation_idx, patients, X, y):
    #train_pat is a df with 2 columns: pat_lis (id), imgXcor (number of images per coronary (LAD,LCX,RCA))
    i = 0
    te = train_pat.iloc[validation_idx].pat_list # id patients training crossvalidation
    im_cor = np.concatenate((train_pat.iloc[validation_idx].imgXcor).reset_index(drop=True)) # train_pat.imgXcor rows corresponding to validation patients  + np.concatenate to make the list of lists (imgXcor) into a single list
    imgXcor = [x for x in im_cor if x != 0] # remove zeros from the list (imgXcor[coronary]=0 if there are no images for such coronary )
    corXpat = [sum(y>0 for y in x) for x in train_pat.iloc[validation_idx].imgXcor] # number of coronaries represented per patient
    y_validation_pat_fold = y_train_pat[validation_idx]
    for pt in te:
        indices_te = [i for i, s in enumerate(patients) if pt in s]# validation indices in X corresponding to selected patient 
        x_rows_cor_te = X.iloc[indices_te]# X rows in validation
        if i == 0:#first patient -> initialize
            x_validation_fold = x_rows_cor_te
            i = 1
        else:  # from the second patient on -> concatente   
            x_validation_fold = pd.concat([x_validation_fold,x_rows_cor_te]) 

    return(x_validation_fold,y_validation_pat_fold,imgXcor,corXpat)



## Majority Voting

In [ ]:
def major_voting(y_pred, y_prob, imgXcor, n_classes): 
    y_pred_cor_mv = []
    y_prob_cor_mean_sum = []
    y_prob_cor_mean_0 = []
    y_prob_cor_mean_1 = []
    y_prob_cor_mean_2 = []

    n_img = 0
    for n in imgXcor: 
        if n_img < sum(imgXcor):
            # predictions
            y_pred_cor = y_pred[n_img : n_img + n] # predictions corresponding to the coronary considered
            numerosity_classes = []
            for i in range(n_classes-1, -1, -1):
                numerosity_classes.append((y_pred_cor == i).sum()) # count number of predictions per class, descendent order of classes such that in case of tie the worst class is selected
            class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
            y_pred_cor_mv.append(class_max)
            
            # prediction probabilities - considera solo le immagini con predizione = class_max
            y_prob_cor = y_prob[n_img:n_img+n, :] # prediction probabilities corresponding to the coronary considered
            
            # Trova gli indici delle immagini che hanno predetto la classe vincente
            winning_indices = np.where(y_pred_cor == class_max)[0]
            
            # Seleziona solo le probabilità delle immagini che hanno predetto la classe vincente
            y_prob_cor_winning = y_prob_cor[winning_indices, :]
            
            # Calcola la media solo su queste immagini per tutte le classi
            y_prob_cor_mean = y_prob_cor_winning.mean(axis=0)
            tot = y_prob_cor_mean.sum() # calculate the total sum of probabilities resulted from mean

            if n_classes > 2: #multiclass
                y_prob_cor_mean_sum.append([x / tot for x in y_prob_cor_mean]) #tutte le colonne
                y_prob_cor_mean_0.append(y_prob_cor_mean[0]/tot)
                y_prob_cor_mean_1.append(y_prob_cor_mean[1]/tot) 
                y_prob_cor_mean_2.append(y_prob_cor_mean[2]/tot)
            else: #binary
                y_prob_cor_mean_sum.append([x / tot for x in y_prob_cor_mean])
                y_prob_cor_mean_0.append(y_prob_cor_mean[0]/tot)
                y_prob_cor_mean_1.append(y_prob_cor_mean[1]/tot) 
                y_prob_cor_mean_2.append(0)  # Placeholder per compatibilità
                                           
            n_img = n_img + n

    return (y_pred_cor_mv, y_prob_cor_mean_sum, y_prob_cor_mean_0, y_prob_cor_mean_1, y_prob_cor_mean_2)

In [ ]:
# Ronchetti
# def major_voting(y_pred,y_prob,imgXcor, n_classes): 
#     y_pred_cor_mv = []
#     y_prob_cor_mean_sum = []
#     y_prob_cor_mean_0 = []
#     y_prob_cor_mean_1 = []
#     y_prob_cor_mean_2 = []

#     n_img = 0
#     for n in imgXcor: 
#         if n_img < sum(imgXcor):
#             # predictions
#             y_pred_cor = y_pred[n_img : n_img + n] # predictions corresponding to the coronary considered
#             numerosity_classes=[]
#             for i in range(n_classes-1, -1,-1):
#                 numerosity_classes.append((y_pred_cor == i).sum()) # count number of predictions per class, descendent order of classes such that in case of tie the worst class is selected
#             class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
#             y_pred_cor_mv.append(class_max)
#             # prediction probabilities
#             y_prob_cor = y_prob[n_img:n_img+n,:] # prediction probabilities corresponding to the coronary considered                      
#             y_prob_cor_mean = y_prob_cor.mean(axis=0)# perform the mean per coronary (and per class)
#             tot = y_prob_cor_mean.sum()# calculate the total sum of probabilities resulted from mean (the sum of prediction probabilities of the classes must be equal to 1)

            
#             if n_classes>2: #multiclass
#                 y_prob_cor_mean_sum.append([x / tot for x in y_prob_cor_mean]) #tutte le colonne
#                 y_prob_cor_mean_0.append(y_prob_cor_mean[0]/tot)
#                 y_prob_cor_mean_1.append(y_prob_cor_mean[1]/tot) 
#                 y_prob_cor_mean_2.append(y_prob_cor_mean[2]/tot)
#             else: #binary
#                 y_prob_cor_mean_sum.append([x / tot for x in y_prob_cor_mean])
#                 y_prob_cor_mean_0.append(y_prob_cor_mean[0]/tot)
#                 y_prob_cor_mean_1.append(y_prob_cor_mean[1]/tot) 
                                           
#             n_img = n_img + n

#     return (y_pred_cor_mv,y_prob_cor_mean_sum,y_prob_cor_mean_0,y_prob_cor_mean_1,y_prob_cor_mean_2)


In [ ]:
def calculate_specificity(true_labels, predicted_labels, n_classes):
    if n_classes > 2: # multiclass task
        specificity_scores = []
        totTP = 0
        totTN = 0
        totFP = 0
        totFN = 0
        for i in range(n_classes):
            cm = confusion_matrix(true_labels, predicted_labels, labels=np.arange(0, n_classes, step=1))
            tp = cm[i,i]
            fn = sum(cm[i,:])-tp
            fp = sum([c[i] for c in cm]) - tp
            tn = (sum(sum(cm))-(tp+fn+fp))
            specificity_scores.append(tn / (tn + fp) if (tn + fp) != 0 else 0)

            totTP = totTP + cm[i,i]
            totFN = totFN + sum(cm[i,:])-tp
            totFP = totFP + sum([c[i] for c in cm]) - tp
            totTN = totTN + (sum(sum(cm))-(tp+fn+fp))

        specificity = sum(specificity_scores) / n_classes
        specificity_micro = totTN/(totTN+totFP)
    else: # binary task
        tn, fp, *_ = confusion_matrix(true_labels, predicted_labels, labels=[0,1]).ravel()
        specificity_micro = np.nan
        specificity = tn / (tn + fp)
    
    
    return specificity, specificity_micro

## Worst Coronaria

### Binary

In [ ]:
def worst_CR_pat_binary(corXpat, y_pred_cor_mv, y_prob_cor_mv, y_prob_cor_mv_0, y_prob_cor_mv_1):
    """
    Calcola il worst case per paziente e la probabilità come:
    1 - (media delle due classi con probabilità più basse delle coronarie che hanno predetto la classe finale)
    """
    n_cor = 0
    y_pred_pat = []
    y_prob_pat_final = []
    y_prob_pat_0 = []
    y_prob_pat_1 = []
    
    for n in corXpat:
        # Predizioni delle coronarie del paziente
        y_pred_mjv_cor = y_pred_cor_mv[n_cor : n_cor + n]
        
        # Trova la classe peggiore (worst case)
        pred_mjv_cor_pat = max(y_pred_mjv_cor)
        y_pred_pat.append(pred_mjv_cor_pat)
        
        # Trova le coronarie che hanno predetto la classe finale
        indices_matching_final = []
        for i, pred in enumerate(y_pred_mjv_cor):
            if pred == pred_mjv_cor_pat:
                indices_matching_final.append(n_cor + i)
        

                    # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
        probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
        probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
        
        # Calcola le medie
        mean_prob_0 = np.mean(probs_class_0)
        mean_prob_1 = np.mean(probs_class_1)
        
        # Per il caso binario, usa la probabilità media della classe finale predetta
        if pred_mjv_cor_pat == 0:
            prob_final = mean_prob_0
        else:  # pred_mjv_cor_pat == 1
            prob_final = mean_prob_1
        
        y_prob_pat_final.append(prob_final)
        y_prob_pat_0.append(mean_prob_0)
        y_prob_pat_1.append(mean_prob_1)
        
        n_cor = n_cor + n
        
    return (y_pred_pat, y_prob_pat_final, y_prob_pat_0, y_prob_pat_1)

### Multiclass

In [ ]:
def worst_CR_pat_multiclass(corXpat, y_pred_cor_mv, y_prob_cor_mv, y_prob_cor_mv_0, y_prob_cor_mv_1, y_prob_cor_mv_2):
    """
    Calcola il worst case per paziente e la probabilità come:
    1 - (media delle due classi con probabilità più basse delle coronarie che hanno predetto la classe finale)
    """
    n_cor = 0
    y_pred_pat = []
    y_prob_pat_final = []
    y_prob_pat_0 = []
    y_prob_pat_1 = []
    y_prob_pat_2 = []
    
    for n in corXpat:
        # Predizioni delle coronarie del paziente
        y_pred_mjv_cor = y_pred_cor_mv[n_cor : n_cor + n]
        
        # Trova la classe peggiore (worst case)
        pred_mjv_cor_pat = max(y_pred_mjv_cor)
        y_pred_pat.append(pred_mjv_cor_pat)
        
        # Trova le coronarie che hanno predetto la classe finale
        indices_matching_final = []
        for i, pred in enumerate(y_pred_mjv_cor):
            if pred == pred_mjv_cor_pat:
                indices_matching_final.append(n_cor + i)
        

        # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
        probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
        probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
        probs_class_2 = [y_prob_cor_mv_2[idx] for idx in indices_matching_final]
        
        # Calcola le medie
        mean_prob_0 = np.mean(probs_class_0)
        mean_prob_1 = np.mean(probs_class_1)
        mean_prob_2 = np.mean(probs_class_2)
        
        # Trova le due classi con probabilità più basse
        class_probs = [mean_prob_0, mean_prob_1, mean_prob_2]
        sorted_probs = sorted(class_probs)
        
        # Calcola 1 - (somma delle due probabilità più basse)
        prob_final = sorted_probs[2]/(sorted_probs[1]+sorted_probs[2])
        
        y_prob_pat_final.append(prob_final)
        y_prob_pat_0.append(mean_prob_0)
        y_prob_pat_1.append(mean_prob_1)
        y_prob_pat_2.append(mean_prob_2)
            
        
        n_cor = n_cor + n
        
    return (y_pred_pat, y_prob_pat_final, y_prob_pat_0, y_prob_pat_1, y_prob_pat_2)

In [ ]:
# def worst_CR_pat(corXpat, y_pred_cor_mv, y_prob_cor_mv, y_prob_cor_mv_0, y_prob_cor_mv_1, y_prob_cor_mv_2):
#     """
#     Calcola il worst case per paziente e la probabilità come:
#     1 - (media delle due classi con probabilità più basse delle coronarie che hanno predetto la classe finale)
#     """
#     n_cor = 0
#     y_pred_pat = []
#     y_prob_pat_final = []
#     y_prob_pat_0 = []
#     y_prob_pat_1 = []
#     y_prob_pat_2 = []
    
#     for n in corXpat:
#         # Predizioni delle coronarie del paziente
#         y_pred_mjv_cor = y_pred_cor_mv[n_cor : n_cor + n]
        
#         # Trova la classe peggiore (worst case)
#         pred_mjv_cor_pat = max(y_pred_mjv_cor)
#         y_pred_pat.append(pred_mjv_cor_pat)
        
#         # Trova le coronarie che hanno predetto la classe finale
#         indices_matching_final = []
#         for i, pred in enumerate(y_pred_mjv_cor):
#             if pred == pred_mjv_cor_pat:
#                 indices_matching_final.append(n_cor + i)
        
#         # Calcola la media delle probabilità solo di quelle coronarie
#         if y_prob_cor_mv_2 is not None:  # multiclass
#             # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
#             probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
#             probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
#             probs_class_2 = [y_prob_cor_mv_2[idx] for idx in indices_matching_final]
            
#             # Calcola le medie
#             mean_prob_0 = np.mean(probs_class_0)
#             mean_prob_1 = np.mean(probs_class_1)
#             mean_prob_2 = np.mean(probs_class_2)
            
#             # Trova le due classi con probabilità più basse
#             class_probs = [mean_prob_0, mean_prob_1, mean_prob_2]
#             sorted_probs = sorted(class_probs)
            
#             # Calcola 1 - (somma delle due probabilità più basse)
#             prob_final = 1 - (sorted_probs[0] + sorted_probs[1])
            
#             y_prob_pat_final.append(prob_final)
#             y_prob_pat_0.append(mean_prob_0)
#             y_prob_pat_1.append(mean_prob_1)
#             y_prob_pat_2.append(mean_prob_2)
            
#         else:  # binary
#             # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
#             probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
#             probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
            
#             # Calcola le medie
#             mean_prob_0 = np.mean(probs_class_0)
#             mean_prob_1 = np.mean(probs_class_1)
            
#             # Per il caso binario, usa la probabilità media della classe finale predetta
#             if pred_mjv_cor_pat == 0:
#                 prob_final = mean_prob_0
#             else:  # pred_mjv_cor_pat == 1
#                 prob_final = mean_prob_1
            
#             y_prob_pat_final.append(prob_final)
#             y_prob_pat_0.append(mean_prob_0)
#             y_prob_pat_1.append(mean_prob_1)
#             y_prob_pat_2.append(0)  # Non utilizzata nel caso binario
        
#         n_cor = n_cor + n
        
#     return (y_pred_pat, y_prob_pat_final, y_prob_pat_0, y_prob_pat_1, y_prob_pat_2)

### Binary - Validation

In [ ]:
def worst_CR_pat_binary_validation(corXpat, y_pred_cor_mv, y_prob_cor_mv, y_prob_cor_mv_0, y_prob_cor_mv_1):
    """
    Calcola il worst case per paziente e restituisce un vettore delle probabilità 
    medie [mean_prob_0, mean_prob_1] dalle coronarie che hanno predetto la classe finale
    """
    n_cor = 0
    y_pred_pat = []
    y_prob_pat_final = []  # Ora conterrà vettori [prob_0, prob_1]
    
    for n in corXpat:
        # Predizioni delle coronarie del paziente
        y_pred_mjv_cor = y_pred_cor_mv[n_cor : n_cor + n]
        
        # Trova la classe peggiore (worst case)
        pred_mjv_cor_pat = max(y_pred_mjv_cor)
        y_pred_pat.append(pred_mjv_cor_pat)
        
        # Trova le coronarie che hanno predetto la classe finale
        indices_matching_final = []
        for i, pred in enumerate(y_pred_mjv_cor):
            if pred == pred_mjv_cor_pat:
                indices_matching_final.append(n_cor + i)
        
        # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
        probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
        probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
        
        # Calcola le medie
        mean_prob_0 = np.mean(probs_class_0)
        mean_prob_1 = np.mean(probs_class_1)
        
        # Restituisci il vettore completo delle probabilità
        prob_vector = [mean_prob_0, mean_prob_1]
        y_prob_pat_final.append(prob_vector)
        
        n_cor = n_cor + n
        
    return (y_pred_pat, y_prob_pat_final)

### Multiclass - Validation

In [ ]:
def worst_CR_pat_multiclass_validation(corXpat, y_pred_cor_mv, y_prob_cor_mv, y_prob_cor_mv_0, y_prob_cor_mv_1, y_prob_cor_mv_2):
    """
    Calcola il worst case per paziente e restituisce la media delle probabilità 
    della classe vincente dalle coronarie che l'hanno predetta
    """
    n_cor = 0
    y_pred_pat = []
    y_prob_pat_final = []
    y_prob_pat_0 = []
    y_prob_pat_1 = []
    y_prob_pat_2 = []
    
    for n in corXpat:
        # Predizioni delle coronarie del paziente
        y_pred_mjv_cor = y_pred_cor_mv[n_cor : n_cor + n]
        
        # Trova la classe peggiore (worst case)
        pred_mjv_cor_pat = max(y_pred_mjv_cor)
        y_pred_pat.append(pred_mjv_cor_pat)
        
        # Trova le coronarie che hanno predetto la classe finale
        indices_matching_final = []
        for i, pred in enumerate(y_pred_mjv_cor):
            if pred == pred_mjv_cor_pat:
                indices_matching_final.append(n_cor + i)
        
        # Calcola la media delle probabilità solo di quelle coronarie
        # Raccogli le probabilità delle coronarie che hanno predetto la classe finale
        probs_class_0 = [y_prob_cor_mv_0[idx] for idx in indices_matching_final]
        probs_class_1 = [y_prob_cor_mv_1[idx] for idx in indices_matching_final]
        probs_class_2 = [y_prob_cor_mv_2[idx] for idx in indices_matching_final]
        
        # Calcola le medie
        mean_prob_0 = np.mean(probs_class_0)
        mean_prob_1 = np.mean(probs_class_1)
        mean_prob_2 = np.mean(probs_class_2)
        
        
        prob_vector = [mean_prob_0, mean_prob_1, mean_prob_2]
        y_prob_pat_final.append(prob_vector)
            

        
        n_cor = n_cor + n
        
    return (y_pred_pat, y_prob_pat_final)

# main

In [ ]:
df = pd.read_csv('features_CADRADS_original_wavelet_exponential.csv', index_col=False)

# list of all patients (id images)
patients = []
coronaries = []
for i in range(np.shape(df)[0]):
    patients.append(f'{df["ID"][i][2]}-{df["ID"][i][8:10]}') 
    coronaries.append(f'{df["ID"][i][2]}-{df["ID"][i][8:10]}-{df["ID"][i][3:6]}') 
df.insert(0,'ID_new', patients)
df.insert(1,'ID_new_cor',coronaries)
df = df.sort_values(by='ID_new_cor')

# list unique patients(251)
pat_list = [*set(patients)]
pat_list.sort()

# label unique patients
y_pat_list = [p[0] for p in pat_list]
y_pat_list = pd.DataFrame(y_pat_list)
y_pat_list.columns = ["CR"]

# number of images per coronary
imgXcor = []
cor = ['LAD','LCX','RCA'] 
for p in pat_list:
    imgXcor_row = []
    for c in cor:
        imgXcor_row.append(sum(f'{p}-{c}' in string for string in coronaries))
    imgXcor.append(imgXcor_row) # list of lists, every sublist tells the number of images for each coronary artery LAD, LCX, RCA (could be also zero)
#df with patient information (col1: id, col2:number of images per coronary)
pat_info = pd.DataFrame({'pat_list':pat_list, 'imgXcor':imgXcor})



## train-test split 

In [ ]:
train_pat, test_pat, y_train_pat, y_test_pat = train_test_split(pat_info, y_pat_list, train_size= 0.9, random_state= 49, stratify= y_pat_list)
y_train_pat = np.concatenate(np.array(y_train_pat, dtype=int)) 
y_test_pat = np.concatenate(np.array(y_test_pat, dtype=int))


In [ ]:
# create training sets for each task

# 0-1234-5
y_train_pat_0_1234_5 = y_train_pat.copy() 
y_train_pat_0_1234_5[(y_train_pat == 2)|(y_train_pat == 3)|(y_train_pat == 4)] = 1
y_train_pat_0_1234_5[y_train_pat == 5] = 2

# 12-34
filterr = (y_train_pat == 1) | (y_train_pat == 2) | (y_train_pat == 3)| (y_train_pat == 4)
train_pat_12_34 = train_pat.loc[filterr]
y_train_pat_12_34 = y_train_pat[filterr]
y_train_pat_12_34[(y_train_pat_12_34 == 1) | (y_train_pat_12_34 == 2)] = 0
y_train_pat_12_34[(y_train_pat_12_34 == 3) | (y_train_pat_12_34 == 4)] = 1

# 1-2
filter12 = (y_train_pat == 1) | (y_train_pat == 2)
train_pat_12 = train_pat.loc[filter12] 
y_train_pat_12 = y_train_pat[filter12]
y_train_pat_12[(y_train_pat_12 == 1)] = 0
y_train_pat_12[(y_train_pat_12 == 2)] = 1

# 3-4
filter34 = (y_train_pat == 3) | (y_train_pat == 4)
train_pat_34 = train_pat.loc[filter34] # 64 righe
y_train_pat_34 = y_train_pat[filter34]
y_train_pat_34[(y_train_pat_34 == 3)] = 0
y_train_pat_34[(y_train_pat_34 == 4)] = 1


In [ ]:
# define classification parameters
classifier = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5, max_depth=2, random_state=0) 
fold = 5
skf = StratifiedKFold(n_splits=fold)

In [ ]:
# best lasso regulation parameter for each task (selected on crossvalidation)
k_0_1234_5 = 0.05
k_12_34 = 0.05
k_12 = 0.05
k_34 = 0.1

## crossvalidation

### Task 0 - 1234 - 5

In [ ]:
#0-1234-5
insight_0_1234_5 = []
CM_mean_0_1234_5 = []
CM_sum_0_1234_5 = []

CM_0_1234_5 = []
cm_sum_0_1234_5 = [[0 for j in range(3)] for i in range(3)]
AUC_ROC_macro_0_1234_5 = []
bal_acc_0_1234_5 = []
f1_macro_0_1234_5 = []
sens_macro_0_1234_5 = []
spec_macro_0_1234_5=[]
AUC_ROC_micro_0_1234_5 = []
f1_micro_0_1234_5 = []
sens_micro_0_1234_5 = []
spec_micro_0_1234_5=[]

for train_0_1234_5_idx, validation_0_1234_5_idx in skf.split(train_pat,y_train_pat_0_1234_5):
    
    #training set crossvalidation
    x_train_fold_0_1234_5,y_train_fold_0_1234_5 = training_set_creation(train_pat,train_0_1234_5_idx,patients_new,X,y)
    
    #modify label
    y_train_fold_0_1234_5.loc[(y_train_fold_0_1234_5 == 2)|(y_train_fold_0_1234_5 == 3)|(y_train_fold_0_1234_5 == 4)] = 1
    y_train_fold_0_1234_5.loc[y_train_fold_0_1234_5 == 5] = 2
    
    #validation set crossvalidation
    x_validation_fold_0_1234_5,y_validation_pat_fold_0_1234_5, imgXcor_0_1234_5, corXpat_0_1234_5 = validation_set_creation (train_pat,y_train_pat_0_1234_5,validation_0_1234_5_idx,patients_new,X,y)
    
    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_train_fold_0_1234_5, x_validation_fold_0_1234_5)                    
    
    #feature selection
    x_train_fold_sel,x_validation_fold_sel, *_ = feat_sel_multi (x_train_fold_scaled, x_validation_fold_scaled, y_train_fold_0_1234_5, k_0_1234_5) 
    
    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_train_fold_0_1234_5)
    
    # fit classifier
    classifier.fit(x_train_fold_resample, y_train_fold_resample)
    
    # prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)
    
    #mjv coronary
    y_pred_cor_0_1234_5, y_prob_cor_0_1234_5,y_prob_cor_class0_0_1234_5,y_prob_cor_class1_0_1234_5,y_prob_cor_class2_0_1234_5 = major_voting(y_pred, y_prob, imgXcor_0_1234_5, 3)
    
    # worst CR patient
    y_pred_pat_0_1234_5,y_prob_pat_0_1234_5 = worst_CR_pat_multiclass_validation(corXpat_0_1234_5,y_pred_cor_0_1234_5,y_prob_cor_0_1234_5,y_prob_cor_class0_0_1234_5,y_prob_cor_class1_0_1234_5,y_prob_cor_class2_0_1234_5)
    
    #metrics 
    cm_sum_0_1234_5 = np.array(cm_sum_0_1234_5) + np.array(confusion_matrix(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5))
    CM_0_1234_5.append(confusion_matrix(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5))
    AUC_ROC_macro_0_1234_5.append(roc_auc_score(y_validation_pat_fold_0_1234_5, y_prob_pat_0_1234_5,average = 'macro',multi_class = 'ovr'))
    AUC_ROC_micro_0_1234_5.append(roc_auc_score(y_validation_pat_fold_0_1234_5, y_prob_pat_0_1234_5,average = 'micro',multi_class = 'ovr'))
    bal_acc_0_1234_5.append(balanced_accuracy_score(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5))
    f1_macro_0_1234_5.append(f1_score(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5, pos_label= 1,average = 'macro'))
    f1_micro_0_1234_5.append(f1_score(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5, pos_label= 1,average = 'micro'))
    sens_macro_0_1234_5.append(recall_score(y_validation_pat_fold_0_1234_5,y_pred_pat_0_1234_5, average='macro'))
    sens_micro_0_1234_5.append(recall_score(y_validation_pat_fold_0_1234_5,y_pred_pat_0_1234_5, average='micro'))
    spec_macro_0_1234_5.append(calculate_specificity(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5, n_classes = 3)[0])
    spec_micro_0_1234_5.append(calculate_specificity(y_validation_pat_fold_0_1234_5, y_pred_pat_0_1234_5, n_classes = 3)[1])

cm_mean = np.stack(CM_0_1234_5).mean(axis=0)
CM_mean_0_1234_5.append(cm_mean)
CM_sum_0_1234_5.append(cm_sum_0_1234_5)
insight_0_1234_5.append([k_0_1234_5,
                         format(np.mean(AUC_ROC_macro_0_1234_5), ".2f"),format(np.std(AUC_ROC_macro_0_1234_5), ".2f"), 
                          format(np.mean(AUC_ROC_micro_0_1234_5), ".2f"),format(np.std(AUC_ROC_micro_0_1234_5), ".2f"), 
                          format(np.mean(bal_acc_0_1234_5), ".2f"),format(np.std(bal_acc_0_1234_5), ".2f"),
                            format(np.mean(f1_macro_0_1234_5), ".2f"),format(np.std(f1_macro_0_1234_5), ".2f"),
                            format(np.mean(f1_micro_0_1234_5), ".2f"),format(np.std(f1_micro_0_1234_5), ".2f"),
                            format(np.mean(sens_macro_0_1234_5), ".2f"),format(np.std(sens_macro_0_1234_5), ".2f"),
                            format(np.mean(sens_micro_0_1234_5), ".2f"),format(np.std(sens_micro_0_1234_5), ".2f"),
                            format(np.mean(spec_macro_0_1234_5), ".2f"),format(np.std(spec_macro_0_1234_5), ".2f"),
                            format(np.mean(spec_micro_0_1234_5), ".2f"),format(np.std(spec_micro_0_1234_5), ".2f")
                            ])
                 
insight_0_1234_5_df = pd.DataFrame(insight_0_1234_5)
insight_0_1234_5_df.columns = ["k","AUC_ROC_macro_mean", "AUC_ROC_macro_std","AUC_ROC_micro_mean", "AUC_ROC_micro_std",
                               "bal_acc_mean", "bal_acc_std",
                               "f1_macro_mean","f1_macro_std","f1_micro_mean","f1_micro_std",
                               "sensitivity_macro_mean","sensitivity_macro_std","sensitivity_micro_mean","sensitivity_micro_std",
                               "specificity_macro_mean","specificity_macro_std","specificity_micro_mean","specificity_micro_std"]

### Task 12 - 34

In [ ]:
#12-34
insight_12_34 = []
CM_mean_12_34  = []
CM_sum_12_34  = []

CM_12_34  = []
cm_sum_12_34  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_12_34 = []
bal_acc_12_34 = []
f1_12_34 = []
sens_12_34=[]
spec_12_34=[]

for train_12_34_idx, validation_12_34_idx in skf.split(train_pat_12_34,y_train_pat_12_34):
    #training set crossvalidation
    x_train_fold_12_34,y_train_fold = training_set_creation(train_pat_12_34,train_12_34_idx,patients_new,X,y)
    #modify label
    y_train_fold_12_34 = y_train_fold.copy()
    y_train_fold_12_34.loc[(y_train_fold == 1) | (y_train_fold == 2)] = 0
    y_train_fold_12_34.loc[(y_train_fold == 3) | (y_train_fold == 4)] = 1
    #validation set crossvalidation
    x_validation_fold_12_34,y_validation_pat_fold_12_34,imgXcor_12_34,corXpat_12_34 = validation_set_creation (train_pat_12_34,y_train_pat_12_34,validation_12_34_idx,patients_new,X,y)                     
    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_train_fold_12_34, x_validation_fold_12_34)                    
    #feature selection
    x_train_fold_sel,x_validation_fold_sel, *_ = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_train_fold_12_34, k_12_34) 
    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_train_fold_12_34)
    # fit classifier
    classifier.fit(x_train_fold_resample, y_train_fold_resample)
    #prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)
    #mjv coronary
    y_pred_cor_12_34, y_prob_cor_mean_sum_12_34,y_prob_cor_class0_12_34,y_prob_cor_class1_12_34,*_ = major_voting(y_pred, y_prob, imgXcor_12_34, 2)
    #worst CR patient
    y_pred_pat_12_34,y_prob_pat_class1_12_34 = worst_CR_pat_binary_validation(corXpat_12_34,y_pred_cor_12_34,y_prob_cor_mean_sum_12_34,y_prob_cor_class0_12_34,y_prob_cor_class1_12_34)
    #metrics
    cm_sum_12_34 = np.array(cm_sum_12_34) + np.array(confusion_matrix(y_validation_pat_fold_12_34, y_pred_pat_12_34))
    CM_12_34.append(confusion_matrix(y_validation_pat_fold_12_34, y_pred_pat_12_34))
    AUC_ROC_12_34.append(roc_auc_score(y_validation_pat_fold_12_34, np.array(y_prob_pat_class1_12_34)[:, 1]))
    bal_acc_12_34.append(balanced_accuracy_score(y_validation_pat_fold_12_34, y_pred_pat_12_34))
    f1_12_34.append((f1_score(y_validation_pat_fold_12_34, y_pred_pat_12_34, pos_label= 1)))
    sens_12_34.append(recall_score(y_validation_pat_fold_12_34,y_pred_pat_12_34))
    spec_12_34.append(calculate_specificity(y_validation_pat_fold_12_34, y_pred_pat_12_34, n_classes = 2)[0])
cm_mean = np.stack(CM_12_34).mean(axis=0)
CM_mean_12_34.append(cm_mean)
CM_sum_12_34.append(cm_sum_12_34)
insight_12_34.append([k_12_34,format(np.mean(AUC_ROC_12_34), ".2f"),format(np.std(AUC_ROC_12_34), ".2f"), 
            format(np.mean(bal_acc_12_34), ".2f"),format(np.std(bal_acc_12_34), ".2f"),
            format(np.mean(f1_12_34), ".2f"),format(np.std(f1_12_34), ".2f"),
            format(np.mean(sens_12_34), ".2f"),format(np.std(sens_12_34), ".2f"),
            format(np.mean(spec_12_34), ".2f"),format(np.std(spec_12_34), ".2f")])
        
insight_12_34_df = pd.DataFrame(insight_12_34)
insight_12_34_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std ", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


### Task 1 - 2

In [ ]:
#1-2
insight_12 = []
CM_mean_12  = []
CM_sum_12  = []

CM_12  = []
cm_sum_12  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_12 = []
bal_acc_12 = []
f1_12 = []
sens_12=[]
spec_12=[]

for train_12_idx, validation_12_idx in skf.split(train_pat_12,y_train_pat_12):
    #training set crossvalidation
    x_train_fold_12,y_train_fold = training_set_creation(train_pat_12,train_12_idx,patients_new,X,y)
    #modify label
    y_train_fold_12 = y_train_fold.copy()
    y_train_fold_12.loc[y_train_fold == 1] = 0
    y_train_fold_12.loc[y_train_fold == 2] = 1
    #validation set crossvalidation
    x_validation_fold_12,y_validation_pat_fold_12,imgXcor_12,corXpat_12 = validation_set_creation (train_pat_12,y_train_pat_12,validation_12_idx,patients_new,X,y)
    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_train_fold_12, x_validation_fold_12)                    
    #feature selection
    x_train_fold_sel,x_validation_fold_sel, *_ = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_train_fold_12, k_12) 
    if len(x_train_fold_sel.columns)!=0:
        #smote
        x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_train_fold_12)
        #fit classifier
        classifier.fit(x_train_fold_resample, y_train_fold_resample)
        #prediction
        y_pred = classifier.predict(x_validation_fold_sel)
        y_prob = classifier.predict_proba(x_validation_fold_sel)
        #mjv coronary
        y_pred_cor_12, y_prob_cor_mean_sum_12,y_prob_cor_class0_12,y_prob_cor_class1_12,*_ = major_voting(y_pred, y_prob, imgXcor_12, 2)
        #worst CR patient
        y_pred_pat_12,y_prob_pat_class1_12 = worst_CR_pat_binary_validation(corXpat_12,y_pred_cor_12,y_prob_cor_mean_sum_12,y_prob_cor_class0_12,y_prob_cor_class1_12)
        #metrics
        cm_sum_12 = np.array(cm_sum_12) + np.array(confusion_matrix(y_validation_pat_fold_12, y_pred_pat_12))
        CM_12.append(confusion_matrix(y_validation_pat_fold_12, y_pred_pat_12))
        AUC_ROC_12.append(roc_auc_score(y_validation_pat_fold_12, np.array(y_prob_pat_class1_12)[:, 1]))
        bal_acc_12.append(balanced_accuracy_score(y_validation_pat_fold_12, y_pred_pat_12))
        f1_12.append((f1_score(y_validation_pat_fold_12, y_pred_pat_12, pos_label= 1)))
        sens_12.append(recall_score(y_validation_pat_fold_12,y_pred_pat_12))
        spec_12.append(calculate_specificity(y_validation_pat_fold_12, y_pred_pat_12, n_classes = 2)[0])
cm_mean = np.stack(CM_12).mean(axis=0)
CM_mean_12.append(cm_mean)
CM_sum_12.append(cm_sum_12)
insight_12.append([k_12,format(np.mean(AUC_ROC_12), ".2f"),format(np.std(AUC_ROC_12), ".2f"), 
            format(np.mean(bal_acc_12), ".2f"),format(np.std(bal_acc_12), ".2f"),
            format(np.mean(f1_12), ".2f"),format(np.std(f1_12), ".2f"),
            format(np.mean(sens_12), ".2f"),format(np.std(sens_12), ".2f"),
            format(np.mean(spec_12), ".2f"),format(np.std(spec_12), ".2f")])

insight_12_df = pd.DataFrame(insight_12)
insight_12_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


### Task 3 - 4

In [ ]:
#3-4
insight_34 = []
CM_mean_34  = []
CM_sum_34  = []

CM_34  = []
cm_sum_34  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_34 = []
bal_acc_34 = []
f1_34 = []
sens_34=[]
spec_34=[]

for train_34_idx, validation_34_idx in skf.split(train_pat_34,y_train_pat_34):
    #training set crossvalidation
    x_train_fold_34,y_train_fold = training_set_creation(train_pat_34,train_34_idx,patients_new,X,y)
    #modify label
    y_train_fold_34 = y_train_fold.copy()
    y_train_fold_34.loc[y_train_fold == 3] = 0
    y_train_fold_34.loc[y_train_fold == 4] = 1
    #validation set crossvalidation
    x_validation_fold_34,y_validation_pat_fold_34,imgXcor_34,corXpat_34 = validation_set_creation (train_pat_34,y_train_pat_34,validation_34_idx,patients_new,X,y)             
    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_train_fold_34, x_validation_fold_34)                    
    #feature selection
    x_train_fold_sel,x_validation_fold_sel, *_ = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_train_fold_34, k_34) 
    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_train_fold_34)
    #fit classifier
    classifier.fit(x_train_fold_resample, y_train_fold_resample)
    #prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)
    #mjv coronary
    y_pred_cor_34, y_prob_cor_mean_sum_34,y_prob_cor_class0_34,y_prob_cor_class1_34,*_ = major_voting(y_pred, y_prob, imgXcor_34, 2)
    #worst CR patient
    y_pred_pat_34,y_prob_pat_class1_34 = worst_CR_pat_binary_validation(corXpat_34,y_pred_cor_34,y_prob_cor_mean_sum_34,y_prob_cor_class0_34,y_prob_cor_class1_34)
    #metrics
    cm_sum_34 = np.array(cm_sum_34) + np.array(confusion_matrix(y_validation_pat_fold_34, y_pred_pat_34))
    CM_34.append(confusion_matrix(y_validation_pat_fold_34, y_pred_pat_34))
    AUC_ROC_34.append(roc_auc_score(y_validation_pat_fold_34, np.array(y_prob_pat_class1_34)[:, 1]))
    bal_acc_34.append(balanced_accuracy_score(y_validation_pat_fold_34, y_pred_pat_34))
    f1_34.append((f1_score(y_validation_pat_fold_34, y_pred_pat_34, pos_label= 1)))
    sens_34.append(recall_score(y_validation_pat_fold_34,y_pred_pat_34))
    spec_34.append(calculate_specificity(y_validation_pat_fold_34, y_pred_pat_34, n_classes = 2)[0])
cm_mean = np.stack(CM_34).mean(axis=0)
CM_mean_34.append(cm_mean)
CM_sum_34.append(cm_sum_34)
insight_34.append([k_34,format(np.mean(AUC_ROC_34), ".2f"),format(np.std(AUC_ROC_34), ".2f"), 
            format(np.mean(bal_acc_34), ".2f"),format(np.std(bal_acc_34), ".2f"),
            format(np.mean(f1_34), ".2f"),format(np.std(f1_34), ".2f"),
            format(np.mean(sens_34), ".2f"),format(np.std(sens_34), ".2f"),
            format(np.mean(spec_34), ".2f"),format(np.std(spec_34), ".2f")])       

insight_34_df = pd.DataFrame(insight_34)
insight_34_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


## model fitting on whole training set + cascade of models on test set

### sets preparation

In [ ]:
################ 0-1234-5
#training
i = 0  
for pt in train_pat.pat_list:
    indices_tr = [i for i, s in enumerate(patients_new) if pt in s] # trainig indices in X and y corresponding to selected patient 
    x_rows_cor_tr = X.iloc[indices_tr] # X rows in training
    y_rows_cor_tr = y[indices_tr] # y rows in training
    if i == 0: # first patient -> initialize
        x_train = x_rows_cor_tr
        y_train = y_rows_cor_tr
        i = 1
    else: # from second patient on -> concatenate
        x_train = pd.concat([x_train,x_rows_cor_tr]) 
        y_train = pd.concat([y_train,y_rows_cor_tr]) 
#modify label
y_train_0_1234_5 = y_train.copy()
y_train_0_1234_5[(y_train==2)|(y_train==3)|(y_train==4)] = 1
y_train_0_1234_5[(y_train==5)] = 2

#test
i = 0  
im_cor_test_0_1234_5 = np.concatenate((test_pat.imgXcor).reset_index(drop=True)) # from list of lists to single list
imgXcor_test_0_1234_5 = [x for x in im_cor_test_0_1234_5 if x != 0] # remove zeros (when a coronary is not represented for such patient)
corXpat_test_0_1234_5 = [sum(y>0 for y in x) for x in test_pat.imgXcor] #number of coronaries per patient with at least one image
for pt in test_pat.pat_list:
    indices_te = [i for i, s in enumerate(patients_new) if pt in s]# test indices in X corresponding to selected patient 
    x_rows_cor_te = X.iloc[indices_te]# X rows in test
    if i == 0:# first patient -> initialize
        x_test = x_rows_cor_te
        i = 1
    else:     # from second patient on -> concatenate
        x_test = pd.concat([x_test,x_rows_cor_te]) 
#modify label
y_pat_test_0_1234_5_global=y_test_pat.copy()
y_pat_test_0_1234_5 = y_test_pat.copy()
y_pat_test_0_1234_5[(y_test_pat==2)|(y_test_pat==3)|(y_test_pat==4)] = 1
y_pat_test_0_1234_5[(y_test_pat==5)] = 2

################### 12-34
#training
i = 0
for pt in train_pat_12_34.pat_list:
    indices = [i for i, s in enumerate(patients_new) if pt in s]# trainig indices in X and y corresponding to selected patient 
    x_rows_cor_tr = X.iloc[indices]# X rows in training
    y_rows_cor_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_12_34 = x_rows_cor_tr
        y_train_12_34 = y_rows_cor_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_12_34 = pd.concat([x_train_12_34,x_rows_cor_tr])
        y_train_12_34 = pd.concat([y_train_12_34,y_rows_cor_tr]) 
# modify label 
y_train_12_34[(y_train_12_34==1)|(y_train_12_34==2)] = 0
y_train_12_34[(y_train_12_34==3)|(y_train_12_34==4)] = 1

################### 1-2
#training
i=0
for pt in train_pat_12.pat_list:
    indices = [i for i, s in enumerate(patients_new) if pt in s]# trainig indices in X and y corresponding to selected patient 
    x_rows_cor_tr = X.iloc[indices]# X rows in training
    y_rows_cor_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_12 = x_rows_cor_tr
        y_train_12 = y_rows_cor_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_12 = pd.concat([x_train_12,x_rows_cor_tr]) 
        y_train_12 = pd.concat([y_train_12,y_rows_cor_tr]) 
#modify label
y_train_12[(y_train_12==1)] = 0
y_train_12[(y_train_12==2)] = 1

################### 3-4
#training
i = 0
for pt in train_pat_34.pat_list:
    indices = [i for i, s in enumerate(patients_new) if pt in s]# trainig indices in X and y corresponding to selected patient 
    x_rows_cor_tr = X.iloc[indices]# X rows in training
    y_rows_cor_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_34 = x_rows_cor_tr
        y_train_34 = y_rows_cor_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_34 = pd.concat([x_train_34,x_rows_cor_tr]) 
        y_train_34 = pd.concat([y_train_34,y_rows_cor_tr]) 
#modify label
y_train_34[(y_train_34==3)] = 0
y_train_34[(y_train_34==4)] = 1

### re-training and test

In [ ]:
############ 0-1234-5
#robust scaler
x_train_scaled, x_test_scaled = scaling_robust (x_train, x_test)    
#feature selection    
x_train_sel,x_test_sel,*_ = feat_sel_multi(x_train_scaled, x_test_scaled, y_train_0_1234_5, k_0_1234_5) 
print(x_train_sel.columns)
#smote
x_train_resample, y_train_resample = resampling_smote (x_train_sel,y_train_0_1234_5)
#classifier fitting
classifier.fit(x_train_resample, y_train_resample)
#prediction
y_pred_test_0_1234_5 = classifier.predict(x_test_sel)
y_prob_test_0_1234_5 = classifier.predict_proba(x_test_sel)
#mjv coronary
y_pred_cor_test_0_1234_5, y_prob_cor_mean_sum_test_0_1234_5, y_prob_cor_class0_test_0_1234_5,y_prob_cor_class1_test_0_1234_5,y_prob_cor_class2_test_0_1234_5= major_voting(y_pred_test_0_1234_5, y_prob_test_0_1234_5, imgXcor_test_0_1234_5, 3)
#worst CR patient
y_pred_pat_test_0_1234_5,y_prob_pat_test_0_1234_5,y_prob_pat_class0_test_0_1234_5,y_prob_pat_class1_test_0_1234_5,y_prob_pat_class2_test_0_1234_5 = worst_CR_pat_multiclass(corXpat_test_0_1234_5,y_pred_cor_test_0_1234_5,y_prob_cor_mean_sum_test_0_1234_5, y_prob_cor_class0_test_0_1234_5,y_prob_cor_class1_test_0_1234_5,y_prob_cor_class2_test_0_1234_5) 

############## 12-34
#test set 12-34 task creation
# take just the patients correctly predicted in 0-1234-5 task and belonging to class 1-2-3-4 ->create 12-34 test set
filter_correct_prediction_12_34 = [np.transpose(y_pred_pat_test_0_1234_5 == np.transpose(y_pat_test_0_1234_5))][0] 
pat_test_set_correct_12_34 = test_pat.loc[filter_correct_prediction_12_34 == True] 
y_pat_test_set_correct = [i for indx,i in enumerate(y_test_pat) if filter_correct_prediction_12_34[indx] == True] 
filter_test_12_34 = [True if x==1 or x==2 or x==3 or x==4 else False for x in y_pat_test_set_correct] 
pat_test_set_12_34 = pat_test_set_correct_12_34.loc[filter_test_12_34] 
y_pat_test_12_34 = [i for indx,i in enumerate(y_pat_test_set_correct) if filter_test_12_34[indx] == True] 
y_pat_test_12_34_binary = [0 if x ==1 or x==2 else 1 if x==3 or x==4 else x for x in y_pat_test_12_34] 
i = 0
im_cor_test_12_34 = np.concatenate((pat_test_set_12_34.imgXcor).reset_index(drop=True))# from list of lists to single list
imgXcor_test_12_34 = [x for x in im_cor_test_12_34 if x != 0] # remove zeros, coronaries not represented for the patient (no images)
corXpat_test_12_34 = [sum(y>0 for y in x) for x in pat_test_set_12_34.imgXcor] # number of coronaries per patient( with at least one image)
for pt_te in pat_test_set_12_34.pat_list:
    indices_te_12_34 = [i for i, s in enumerate(patients_new) if pt_te in s]# test indices in X and y corresponding to selected patient
    x_rows_cor_test = X.iloc[indices_te_12_34]# X rows in test
    if i == 0: # first patient -> initialize
        x_test_set_12_34= x_rows_cor_test
        i = 1
    else:    # from second patient on -> concatenate
        x_test_set_12_34 = pd.concat([x_test_set_12_34,x_rows_cor_test])
#robust scaler
x_train_scaled, x_test_scaled = scaling_robust (x_train_12_34, x_test_set_12_34)        
#feature selection
x_train_sel,x_test_sel,*_ = feat_sel(x_train_scaled, x_test_scaled, y_train_12_34, k_12_34) 
print(x_train_sel.columns)
#smote
x_train_resample, y_train_resample = resampling_smote (x_train_sel, y_train_12_34)
# fit classifier
classifier.fit(x_train_resample, y_train_resample)
#prediction
y_pred_test_12_34 = classifier.predict(x_test_sel)                    
y_prob_test_12_34 = classifier.predict_proba(x_test_sel)
#mjv coronary
y_pred_cor_test_12_34, y_prob_cor_mean_sum_test_12_34,y_prob_cor_class0_test_12_34,y_prob_cor_class1_test_12_34,*_ = major_voting(y_pred_test_12_34, y_prob_test_12_34, imgXcor_test_12_34, 2)
#worst CR patient
y_pred_pat_test_12_34,y_prob_pat_test_12_34,y_prob_pat_class0_test_12_34,y_prob_pat_class1_test_12_34 = worst_CR_pat_binary(corXpat_test_12_34,y_pred_cor_test_12_34,y_prob_cor_mean_sum_test_12_34,y_prob_cor_class0_test_12_34,y_prob_cor_class1_test_12_34)

############## 1-2
#test set 1-2 task creation
# take just the patients correctly predicted in 12-34 task and belonging to class 1-2 create 1-2 test set
filter_correct_prediction_single_class = [np.transpose(y_pred_pat_test_12_34 == np.transpose(y_pat_test_12_34_binary))][0] 
pat_test_set_correct_single_class = pat_test_set_12_34.loc[filter_correct_prediction_single_class == True] 
y_pat_test_set_correct_single_class = [i for indx,i in enumerate(y_pat_test_12_34) if filter_correct_prediction_single_class[indx] == True]
filter_test_12 = [True if x==1 or x==2 else False for x in y_pat_test_set_correct_single_class]
pat_test_set_12 = pat_test_set_correct_single_class.loc[filter_test_12]
if len(pat_test_set_12) != 0:
    y_pat_test_12 = [i for indx,i in enumerate(y_pat_test_set_correct_single_class) if filter_test_12[indx] == True]
    y_pat_test_12_binary = [0 if x == 1 else 1 if x == 2 else x for x in y_pat_test_12]
    i = 0
    im_cor_test_12 = np.concatenate((pat_test_set_12.imgXcor).reset_index(drop=True))# from list of lists to single list
    imgXcor_test_12 = [x for x in im_cor_test_12 if x != 0]# remove zeros, coronaries not represented for the patient (no images)
    corXpat_test_12 = [sum(y>0 for y in x) for x in pat_test_set_12.imgXcor]# number of coronaries per patient( with at least one image)
    for pt_te in pat_test_set_12.pat_list:
        indices_te_12 = [i for i, s in enumerate(patients_new) if pt_te in s]# test indices in X and y corresponding to selected patient
        x_rows_cor_test = X.iloc[indices_te_12]# X rows in test
        if i == 0:# first patient -> initialize
            x_test_set_12= x_rows_cor_test
            i = 1
        else:    # from second patient on -> concatenate
            x_test_set_12 = pd.concat([x_test_set_12,x_rows_cor_test]) 
    #robust scaler
    x_train_scaled12, x_test_scaled12 = scaling_robust (x_train_12, x_test_set_12)        
    #feature selection
    x_train_sel12,x_test_sel12,*_ = feat_sel(x_train_scaled12, x_test_scaled12, y_train_12, k_12) 
    print(x_train_sel12.columns)
    #smote
    x_train_resample12, y_train_resample12 = resampling_smote (x_train_sel12, y_train_12)
    # fit classifier
    classifier.fit(x_train_resample12, y_train_resample12)
    # prediction
    y_pred_test_12 = classifier.predict(x_test_sel12)                    
    y_prob_test_12 = classifier.predict_proba(x_test_sel12)
    # mjv coronary
    y_pred_cor_test_12, y_prob_cor_mean_sum_test_12,y_prob_cor_class0_test_12,y_prob_cor_class1_test_12,*_ = major_voting(y_pred_test_12, y_prob_test_12, imgXcor_test_12, 2)
    # worst CR patient
    y_pred_pat_test_12,y_prob_pat_test_12,y_prob_pat_class0_test_12,y_prob_pat_class1_test_12 = worst_CR_pat_binary(corXpat_test_12,y_pred_cor_test_12,y_prob_cor_mean_sum_test_12,y_prob_cor_class0_test_12,y_prob_cor_class1_test_12)

##### 3-4
#test set 3-4 task creation
# take just the patients correctly predicted in 12-34 task and belonging to class 3-4 -> create 3-4 test set
filter_test_34 = [True if x==3 or x==4 else False for x in y_pat_test_set_correct_single_class]
pat_test_set_34 = pat_test_set_correct_single_class.loc[filter_test_34]
y_pat_test_34 = [i for indx,i in enumerate(y_pat_test_set_correct_single_class) if filter_test_34[indx] == True]
y_pat_test_34_binary = [0 if x == 3 else 1 if x == 4 else x for x in y_pat_test_34]
i = 0
im_cor_test_34 = np.concatenate((pat_test_set_34.imgXcor).reset_index(drop=True))# from list of lists to single list
imgXcor_test_34 = [x for x in im_cor_test_34 if x != 0] # remove zeros, coronaries not represented for the patient (no images)
corXpat_test_34 = [sum(y>0 for y in x) for x in pat_test_set_34.imgXcor]# number of coronaries per patient( with at least one image)
for pt_te in pat_test_set_34.pat_list:
    indices_te_34 = [i for i, s in enumerate(patients_new) if pt_te in s]# test indices in X and y corresponding to selected patient
    x_rows_cor_test = X.iloc[indices_te_34]# X rows in test
    if i == 0:# first patient -> initialize
        x_test_set_34= x_rows_cor_test
        i = 1
    else:    # from second patient on -> concatenate
        x_test_set_34 = pd.concat([x_test_set_34,x_rows_cor_test]) 
#robust scaler
x_train_scaled34, x_test_scaled34 = scaling_robust (x_train_34, x_test_set_34)        
#feature selection
x_train_sel34,x_test_sel34,*_ = feat_sel(x_train_scaled34, x_test_scaled34, y_train_34, k_34) 
print(x_train_sel34.columns)
#smote
x_train_resample34, y_train_resample34 = resampling_smote (x_train_sel34, y_train_34)
# fit classifier
classifier.fit(x_train_resample34, y_train_resample34)
# prediction
y_pred_test_34 = classifier.predict(x_test_sel34)                    
y_prob_test_34 = classifier.predict_proba(x_test_sel34)
# mjv coronary
y_pred_cor_test_34, y_prob_cor_mean_sum_test_34,y_prob_cor_class0_test_34,y_prob_cor_class1_test_34,*_ = major_voting(y_pred_test_34, y_prob_test_34, imgXcor_test_34, 2)
# worst CR patient
y_pred_pat_test_34,y_prob_pat_test_34,y_prob_pat_class0_test_34,y_prob_pat_class1_test_34 = worst_CR_pat_binary(corXpat_test_34,y_pred_cor_test_34,y_prob_cor_mean_sum_test_34,y_prob_cor_class0_test_34,y_prob_cor_class1_test_34)
